# ERA5 sea-ice profiles

Mean ERA5 temperature and humidity profiles north of 60° N for 1980–2019, binned by season, five-year period and sea-ice cover. The result, `data/era5_sea_ice_to_libradtran.nc`, is read by {doc}`sea_ice_era5_plotting` (trends) and {doc}`era5_seasonal_sea_ice_profiles` (atmospheres for pyRadtran). It needs `gcsfs` and network access.

## Data and sampling

The source is the public ARCO-ERA5 archive on Google Cloud Storage, on its coarsest grid (64 × 32 points, 5.6°, 6-hourly, 13 pressure levels), read anonymously. The archive is stored in blocks of 100 time steps (25 days). To keep the download near 2 GB, the notebook reads every fourth block, 25 days out of every 100. Because 100 days does not divide the year, the sampled days move through the calendar and all seasons are covered evenly. The download takes a few minutes and is cached in `data/era5_arctic_6h_64x32_subset.nc` (about 600 MB, not under version control).

The pole row (90° N) is dropped: on this grid it repeats one value 64 times.

In [1]:
from pathlib import Path

import gcsfs
import numpy as np
import pandas as pd
import xarray as xr

CACHE = Path("data/era5_arctic_6h_64x32_subset.nc")
OUTPUT = Path("data/era5_sea_ice_to_libradtran.nc")
URL = "gs://gcp-public-data-arco-era5/ar/1959-2022-6h-64x32_equiangular_with_poles_conservative.zarr"

~/pyRadtran/.scratch/nbextra/google/api_core/_python_version_support.py:212: FutureWarning: You are using a non-supported Python version (3.10.12). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
~/pyRadtran/.scratch/nbextra/google/api_core/_python_version_support.py:212: FutureWarning: You are using a non-supported Python version (3.10.12). Google will not post any further updates to google.cloud._storage_v2 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud._storage_v2.
  warnings.warn(message, FutureWarning)
~/pyRadtran/.scratch/nbextra/google/api_core/_python_version_support.py:212: FutureWarning: You are using a non-supported Python version (3.10.12). Google will not post any further updates to google.cloud.storage_con

In [2]:
if CACHE.exists():
    subset = xr.open_dataset(CACHE).load()
else:
    gcs = gcsfs.GCSFileSystem(token="anon")
    era5 = xr.open_zarr(gcs.get_mapper(URL), chunks={})
    block = era5.temperature.encoding["chunks"][0]  # time steps per stored block
    i0, i1 = np.searchsorted(era5.time.values, np.datetime64("1980-01-01")), np.searchsorted(era5.time.values, np.datetime64("2020-01-01"))
    steps = [i for i in range(i0, i1) if (i // block) % 4 == 0]
    subset = (
        era5[["temperature", "specific_humidity", "sea_ice_cover"]]
        .isel(time=steps)
        .sel(latitude=slice(60, 90))
        .load()
    )
    CACHE.parent.mkdir(exist_ok=True)
    subset.to_netcdf(CACHE)

subset = subset.sel(latitude=slice(60, 88))  # drop the pole row
print(f"{subset.time.size} time steps, {subset.time.dt.year.min().item()}–{subset.time.dt.year.max().item()}, latitudes {subset.latitude.values.round(1)}")

~/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


14600 time steps, 1980–2019, latitudes [61.  66.8 72.6 78.4 84.2]


## Binning

Every grid cell and time step is assigned to a bin of season, five-year period (1980–1984, …, 2015–2019) and sea-ice cover (0–0.1, …, 0.9–1; the first bin includes open water, land cells have no sea-ice cover and drop out). The mean profile of each bin is a plain average over grid cells, not area-weighted. `n` counts the samples per bin.

In [3]:
seasons = ["DJF", "MAM", "JJA", "SON"]
period_start = np.arange(1980, 2020, 5)

season_i = xr.DataArray(pd.Index(seasons).get_indexer(subset.time.dt.season.values), dims="time")
period_i = (subset.time.dt.year - 1980) // 5
sic_i = np.clip(np.floor(subset.sea_ice_cover * 10), 0, 9)  # NaN over land
key = (season_i * 80 + period_i * 10 + sic_i).rename("bin")

all_bins = np.arange(len(seasons) * len(period_start) * 10)
means = subset[["temperature", "specific_humidity"]].groupby(key).mean().reindex(bin=all_bins)
counts = key.groupby(key).count().reindex(bin=all_bins, fill_value=0)

labels = pd.MultiIndex.from_product(
    [seasons, [pd.Timestamp(f"{y + 2}-07-02") for y in period_start], np.round(np.arange(0.05, 1, 0.1), 2)],
    names=["season", "valid_time", "sea_ice_cover"],
)
profiles = (
    xr.Dataset({"t": means.temperature, "q": means.specific_humidity, "n": counts})
    .assign_coords(bin=labels)
    .unstack("bin")
    .rename(level="pressure_level")
    .sel(season=seasons)
    .transpose("season", "valid_time", "sea_ice_cover", "pressure_level")
)
profiles["t"].attrs = {"units": "K", "long_name": "temperature"}
profiles["q"].attrs = {"units": "kg kg-1", "long_name": "specific humidity"}
profiles["n"].attrs = {"long_name": "number of grid-cell samples"}
profiles["pressure_level"].attrs["units"] = "hPa"
profiles["valid_time"].attrs["long_name"] = "middle of the five-year period"
profiles.attrs["source"] = f"ARCO-ERA5 {URL.rsplit('/', 1)[-1]}, every fourth 25-day block, 60-88 N"

profiles.to_netcdf(OUTPUT)
profiles

/tmp/ipykernel_2883053/601372359.py:18: FutureWarning: the `pandas.MultiIndex` object(s) passed as 'bin' coordinate(s) or data variable(s) will no longer be implicitly promoted and wrapped into multiple indexed coordinates in the future (i.e., one coordinate for each multi-index level + one dimension coordinate). If you want to keep this behavior, you need to first wrap it explicitly using `mindex_coords = xarray.Coordinates.from_pandas_multiindex(mindex_obj, 'dim')` and pass it as coordinates, e.g., `xarray.Dataset(coords=mindex_coords)`, `dataset.assign_coords(mindex_coords)` or `dataarray.assign_coords(mindex_coords)`.
  xr.Dataset({"t": means.temperature, "q": means.specific_humidity, "n": counts})


<xarray.Dataset> Size: 36kB
Dimensions:         (season: 4, valid_time: 8, sea_ice_cover: 10,
                     pressure_level: 13)
Coordinates:
  * season          (season) object 32B 'DJF' 'MAM' 'JJA' 'SON'
  * valid_time      (valid_time) datetime64[ns] 64B 1982-07-02 ... 2017-07-02
  * sea_ice_cover   (sea_ice_cover) float64 80B 0.05 0.15 0.25 ... 0.85 0.95
  * pressure_level  (pressure_level) int64 104B 50 100 150 200 ... 850 925 1000
Data variables:
    t               (season, valid_time, sea_ice_cover, pressure_level) float32 17kB ...
    q               (season, valid_time, sea_ice_cover, pressure_level) float32 17kB ...
    n               (season, valid_time, sea_ice_cover) int64 3kB 17617 ... 3...
Attributes:
    source:   ARCO-ERA5 1959-2022-6h-64x32_equiangular_with_poles_conservativ...

Samples per bin in winter. Each bin holds at least a thousand grid-cell samples; the open-water and fully ice-covered bins hold most of them.

In [4]:
profiles.n.sel(season="DJF").to_pandas()

sea_ice_cover,0.05,0.15,0.25,0.35,0.45,0.55,0.65,0.75,0.85,0.95
valid_time,,,,,,,,,,
1982-07-02,17617,2574,2737,2340,1788,1684,2316,3646,11592,86186
1987-07-02,20448,3343,2815,2508,1775,1610,2380,4473,10781,96699
1992-07-02,15999,2378,2012,1799,1574,1550,2045,2673,7886,74692
1997-07-02,14977,2185,2228,2090,1513,1187,1765,2828,7441,70874
2002-07-02,19134,3369,2459,1846,1755,1825,2424,3984,10007,77949
2007-07-02,27763,3789,3365,2766,3013,2960,2930,4244,12867,96383
2012-07-02,18340,2544,2490,2126,2122,1759,2268,3340,7656,62235
2017-07-02,20797,3453,2244,1793,1942,1929,2427,3383,8636,66004
